In [58]:
import pandas as pd
from time import time

In [ ]:
import os
from langchain_ollama import OllamaLLM
from langsmith import traceable

QWEN2_5_MODEL = "qwen2.5:3b-instruct"
GEMMA3_27B_MODEL = "gemma3:27b"

# Configure LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "REMOVED_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "ollama-playground"

# Initialize model
# Initialize model with optimizations
model = OllamaLLM(
    model=QWEN2_5_MODEL,
    base_url="http://localhost:11434",  # Explicit URL
    temperature=0.0,                    # Control randomness
    num_predict=256,                    # Limit output length
    top_k=10,                          # Faster sampling
    top_p=0.9,                         # Faster sampling
    repeat_penalty=1.1,                # Prevent repetition
    timeout=30,                        # Reasonable timeout
)

In [60]:
# Get the absolute path of the script
current_dir = os.getcwd()

# Move up two levels to the project root
project_root = os.path.abspath(os.path.join(current_dir, "../.."))

# Construct the CSV path
csv_path = os.path.join(project_root, "data", "reviews_labeled_cleaned.csv")

In [61]:
df = pd.read_csv(csv_path)
df.head()

,source,size,comment,original,Satisfaction,Authenticity,Type,needs_response,Category,Impact,Language,Emotion,cleaned
0,Friedom,short,Not salty enough,Not salty enough,3,Legitimate,Complaint,Yes,Product Quality / Food Quality,Moderate,English,Disappointment,Not salty enough
1,Friedom,short,Overall good chicken and fries,Overall good chicken and fries,4,Legitimate,Praise,No,Product Quality / Food Quality,No issue,English,Satisfaction,Overall good chicken and fries
2,Friedom,medium,Over rated\nTiles that are sticky to each othe...,Over rated\nطوابل لازقين فبعضاهم و مزيتتين ميش...,1,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,"English, Derja",Frustration,Over rated طوابل لازقين فبعضاهم و مزيتتين ميش ...
3,Friedom,short,Food took +30 minutes to come.,Food took +30 minutes to come.,2,Legitimate,Complaint,Yes,Delivery / Logistics,Low Impact,English,Disappointment,Food took 30 minutes to come.
4,Friedom,long,"●●●●Food : ● the buns are so soggy ""DEAD"" like...","●●●●Food : ● the buns are so soggy ""DEAD"" like...",2,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,English,Anger,Food the buns are so soggy DEAD like they cant...


In [62]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["text"],
    template="""
    You are a bot that is expert in languages. your tast is to detect the lannguage of the following text: '{text}'.
    You should not output any other text, just the name of the language using the follwing codes: 
    "English": "en",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Italian": "it",
    "Dutch": "nl",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Chinese": "zh",
    "Korean": "ko",
    "Arabic": "ar",
    "Turkish": "tr",
    "Derja": "dz",
    """,
)


def structure_output(output: str) -> str:
    """Format the output to ensure it is a single language code."""
    result =  output.strip().lower()
    if result in ["en", "fr", "es", "de", "it", "nl", "pt", "ru", "ja", "zh", "ko", "ar", "tr", "dz"]:
        return result
    # if for example it has "ar" than it is arabic and we return "ar" etc
    if "ar" in result:
        return "ar"
    if "fr" in result:
        return "fr"
    if "es" in result:
        return "es"
    if "de" in result:
        return "de"
    if "en" in result:
        return "en"
    
from langchain_core.runnables import RunnableLambda
chain = template | model | StrOutputParser() | RunnableLambda(structure_output)

In [63]:
# Define a traceable function to process the DataFrame
@traceable
def process_dataframe(df):
    results = []
    for index, row in df.iterrows():
        text = row['original']
        start_time = time()
        response = chain.invoke({"text": text})
        end_time = time()
        results.append({
            "index": index,
            "text": text,
            "Language": row['Language'],
            "language_predicted": response,
            "time_taken": end_time - start_time
        })
    return pd.DataFrame(results)

results_df = process_dataframe(df)

In [69]:
lang_name_to_code = {
    "English": "en",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Italian": "it",
    "Dutch": "nl",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Chinese": "zh",
    "Korean": "ko",
    "Arabic": "ar",
    "Turkish": "tr",
    "Derja": "dz",
}

def lang_name_to_code_func(lang_name):
    languages = lang_name.split(',')
    languages = [lang.strip() for lang in languages]
    return ','.join([lang_name_to_code[lang_name] for lang_name in languages])


results_df["Language"] = results_df["Language"].apply(lang_name_to_code_func)

KeyError: 'en'

In [65]:
def score(predicted_language, ground_truth_languages):
    ''' Language detection score calculation
    1: if the two columns match
    0.5: if the predicted_language is in the list of ground truth
    0: otherwise
    '''
    truth_language_list = ground_truth_languages.split(',')
    
    if predicted_language in truth_language_list:
        return 1 if len(truth_language_list) == 1 else 0.5
    else:
        return 0
    
results_df["score"] = results_df.apply(
    lambda row: score(row["language_predicted"], row["Language"]), axis=1
)

In [66]:
results_df["score"].mean()

0.9243781094527364

In [67]:
results_df.groupby("Language").agg({"Language":"count", "score": "sum"})

,Language,score
Language,,
ar,230,224.0
"ar,dz",12,6.0
"ar,dz,fr",1,0.5
"ar,en",3,1.0
"ar,fr",2,1.0
dz,41,28.0
"dz,ar",6,3.0
"dz,fr",4,2.0
en,236,222.0


In [70]:
results_df[results_df["score"] == 0].head(30)

,index,text,Language,language_predicted,time_taken,score
8,8,Service rapide ✅\nMakla bnina ✅\nLes sauces ch...,en,dz,0.659508,0.0
31,31,Je pense que c'est le meilleur dans cette spéc...,fr,dz,0.810484,0.0
51,51,شحال سعر توصيل ؟,ar,dz,0.806315,0.0
55,55,خدمات كي الشعر\nحاسبين الشركة تاع باباكم و حنا...,dz,ar,0.807721,0.0
68,68,Professionelle,fr,de,0.766138,0.0
85,85,Nuuuuuuuuul 000000000,fr,dz,0.786986,0.0
87,87,Service null,ar,fr,0.789582,0.0
106,106,Deplorable service employees arrive at any tim...,en,fr,0.794158,0.0
116,116,"Fast shipping, but the character of the recept...",en,es,0.789056,0.0
128,128,Service 000,fr,en,0.785366,0.0
